# RoadGuard AI — Model Training, Edge Optimization & ONNX Export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahadeab-ai/roadguard-ai/blob/main/ai/colab/02_model_training_and_export.ipynb)

> **Objective**: Train a real-time YOLO road hazard detector, export to ONNX (opset 17), and apply INT8 dynamic quantization for edge mobile deployment.
>
> **Hardware Compliance**: All training and quantization benchmarks execute exclusively on Google Colab Cloud GPUs (T4 / A100). **Zero local GPU computation.**
>
> **Target Taxonomy**:
- `0: D00_longitudinal_crack` (RDD2022 D00)
- `1: D10_transverse_crack` (RDD2022 D10)
- `2: D20_alligator_crack` (RDD2022 D20)
- `3: D40_pothole` (RDD2022 D40)


## 1. Hardware Inspection & Google Drive Checkpoint Mount

In [ ]:
import os
import sys
import shutil
import time
from pathlib import Path
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")

# Check CUDA GPU availability
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[SUCCESS] Cloud GPU Detected: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[WARNING] No GPU found. Ensure Runtime > Change runtime type > T4 GPU is selected.")

# Mount Google Drive for automatic model checkpointing
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/RoadGuard_Models/v1.0.0')
    DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive Checkpoint Directory: {DRIVE_CHECKPOINT_DIR}")
except Exception as e:
    print(f"Google Drive mount skipped: {e}")
    DRIVE_CHECKPOINT_DIR = Path('/content/roadguard_checkpoints')
    DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Install Required Dependencies (Ultralytics, ONNX, OnnxSlim, OnnxRuntime)

In [ ]:
!pip install -q "ultralytics>=8.2.0" "onnx>=1.16.0" "onnxruntime>=1.18.0" "onnxslim>=0.1.28" pyyaml matplotlib

import yaml
from ultralytics import YOLO
import onnx
import onnxruntime as ort
import numpy as np
print("All ML & Export libraries installed successfully.")

## 3. Verify Dataset Binding & Directory Configuration

In [ ]:
DATA_YAML_PATH = Path('/content/roadguard_rdd2022/data.yaml')

# If data.yaml does not exist from notebook 01, create verified configuration
if not DATA_YAML_PATH.exists():
    print(f"{DATA_YAML_PATH} not found. Configuring standard YOLO data descriptor...")
    DATA_YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    dataset_config = {
        'path': '/content/roadguard_rdd2022',
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': 4,
        'names': {0: 'D00_longitudinal_crack', 1: 'D10_transverse_crack', 2: 'D20_alligator_crack', 3: 'D40_pothole'}
    }
    with open(DATA_YAML_PATH, 'w') as f:
        yaml.dump(dataset_config, f, sort_keys=False)

with open(DATA_YAML_PATH) as f:
    loaded_cfg = yaml.safe_load(f)
print("Active Dataset Binding:")
print(yaml.dump(loaded_cfg, indent=2))


## 4. Initialize YOLO Architecture & Hyperparameters

In [ ]:
# Set deterministic seed for complete experimental reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Load baseline YOLOv8n (Nano) targeting edge smartphones (<3.2M params)
model = YOLO('yolov8n.pt')
print(f"Loaded base model: yolov8n.pt")

# Training hyperparameter configuration (ADR 002)
TRAIN_PARAMS = {
    'data': str(DATA_YAML_PATH),
    'epochs': 75,
    'imgsz': 640,
    'batch': 32,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 4,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'cos_lr': True,
    'close_mosaic': 10,
    'patience': 15,
    # Driving-scene augmentations
    'hsv_h': 0.015,
    'hsv_s': 0.700,
    'hsv_v': 0.400,
    'degrees': 3.0,
    'translate': 0.100,
    'scale': 0.500,
    'perspective': 0.0005,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 1.0,
    'mixup': 0.15,
    'project': '/content/runs/train',
    'name': 'roadguard_yolov8n',
    'exist_ok': True
}
print("Training configuration prepared.")

## 5. Execute Cloud Model Training (Colab GPU)

In [ ]:
print("Starting YOLOv8n fine-tuning on Colab GPU...")
start_time = time.time()

# Execute training
results = model.train(**TRAIN_PARAMS)

elapsed = (time.time() - start_time) / 60.0
print(f"\nTraining complete in {elapsed:.2f} minutes.")

BEST_PT_PATH = Path('/content/runs/train/roadguard_yolov8n/weights/best.pt')
print(f"Best PyTorch weights located at: {BEST_PT_PATH}")

# Backup best checkpoint to Google Drive
if BEST_PT_PATH.exists():
    shutil.copy2(BEST_PT_PATH, DRIVE_CHECKPOINT_DIR / 'best.pt')
    print(f"Backed up best.pt to Google Drive: {DRIVE_CHECKPOINT_DIR / 'best.pt'}")

## 6. Comprehensive Validation & Metric Logging

In [ ]:
# Load best checkpoint for validation
best_model = YOLO(str(BEST_PT_PATH))
val_metrics = best_model.val(data=str(DATA_YAML_PATH), split='val', imgsz=640, conf=0.25, iou=0.50)

print("\n--- Validation Metrics ---")
print(f"mAP@50:    {val_metrics.box.map50:.4f}")
print(f"mAP@50-95: {val_metrics.box.map:.4f}")
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")

# Save evaluation summary to JSON
metrics_dict = {
    'mAP50': float(val_metrics.box.map50),
    'mAP50_95': float(val_metrics.box.map),
    'precision': float(val_metrics.box.mp),
    'recall': float(val_metrics.box.mr),
    'classes': {
        'pothole': {'id': 0},
        'road_crack': {'id': 1}
    }
}
with open(DRIVE_CHECKPOINT_DIR / 'validation_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print("Validation metrics written to JSON.")


## 7. Export Model to ONNX (Opset 17, Dynamic Batch, OnnxSlim)

In [ ]:
print("Exporting PyTorch model to ONNX...")

# Export with dynamic batch axis and opset=17
exported_onnx = best_model.export(
    format='onnx',
    opset=17,
    dynamic=True,
    simplify=True,
    imgsz=640
)

FP32_ONNX_PATH = Path('/content/runs/train/roadguard_yolov8n/weights/best.onnx')
if not FP32_ONNX_PATH.exists() and Path(exported_onnx).exists():
    FP32_ONNX_PATH = Path(exported_onnx)

print(f"FP32 ONNX Model generated at: {FP32_ONNX_PATH}")

# Inspect ONNX Model I/O signatures
onnx_model = onnx.load(str(FP32_ONNX_PATH))
onnx.checker.check_model(onnx_model)
print("ONNX model structural integrity: VERIFIED")

session = ort.InferenceSession(str(FP32_ONNX_PATH))
for inp in session.get_inputs():
    print(f"Input:  name='{inp.name}', shape={inp.shape}, type={inp.type}")
for out in session.get_outputs():
    print(f"Output: name='{out.name}', shape={out.shape}, type={out.type}")

## 8. Edge Optimization: Dynamic INT8 Quantization

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

INT8_ONNX_PATH = Path('/content/roadguard-yolo-v1.1.0-int8.onnx')

print("Applying Dynamic INT8 Quantization (weights quantized to INT8)...")
quantize_dynamic(
    model_input=str(FP32_ONNX_PATH),
    model_output=str(INT8_ONNX_PATH),
    weight_type=QuantType.QInt8,
    op_types_to_quantize=['MatMul', 'Conv', 'Gemm']
)

fp32_size_mb = os.path.getsize(str(FP32_ONNX_PATH)) / (1024**2)
int8_size_mb = os.path.getsize(str(INT8_ONNX_PATH)) / (1024**2)
compression = fp32_size_mb / int8_size_mb

print(f"\n--- Model Size Comparison ---")
print(f"FP32 Model Size: {fp32_size_mb:.2f} MB")
print(f"INT8 Model Size: {int8_size_mb:.2f} MB")
print(f"Compression Ratio: {compression:.2f}x reduction")


## 9. Comparative Benchmarking (Latency Profile on CPU/GPU)

In [ ]:
# Benchmark inference latency with dummy 640x640 frame
dummy_input = np.random.randn(1, 3, 640, 640).astype(np.float32)

# CPU session benchmark (simulates mobile host)
cpu_opts = ort.SessionOptions()
cpu_opts.intra_op_num_threads = 2
sess_int8_cpu = ort.InferenceSession(str(INT8_ONNX_PATH), cpu_opts, providers=['CPUExecutionProvider'])
input_name = sess_int8_cpu.get_inputs()[0].name

# Warmup
for _ in range(10):
    _ = sess_int8_cpu.run(None, {input_name: dummy_input})

# Benchmark 50 iterations
latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    _ = sess_int8_cpu.run(None, {input_name: dummy_input})
    latencies.append((time.perf_counter() - t0) * 1000.0)

mean_lat = np.mean(latencies)
p95_lat = np.percentile(latencies, 95)
print(f"\n--- Latency Benchmark (CPU / ARM Simulation) ---")
print(f"Mean Latency: {mean_lat:.2f} ms/frame ({1000.0/mean_lat:.1f} FPS)")
print(f"P95 Latency:  {p95_lat:.2f} ms/frame")
print(f"Sub-80ms Target Met: {p95_lat < 80.0}")

## 10. Checkpoint & Asset Persistence

In [ ]:
# Copy production INT8 ONNX model to Google Drive
try:
    shutil.copy2(INT8_ONNX_PATH, DRIVE_CHECKPOINT_DIR / 'roadguard-yolo-v1.1.0-int8.onnx')
    shutil.copy2(FP32_ONNX_PATH, DRIVE_CHECKPOINT_DIR / 'roadguard-yolo-v1.0.0-fp32.onnx')
    print(f"Successfully saved models to Google Drive: {DRIVE_CHECKPOINT_DIR}")
except Exception as e:
    print(f"Google Drive backup skipped: {e}")

print("\n===================================================")
print(" ROADGUARD AI MODEL TRAINING & EXPORT COMPLETED!")
print("===================================================")
